In [1]:
tabla='ctaam10'
col_tabla = "atenambatenfec"
dat= "dat_cex002_essi"
col_essi='ate_fec'
essi='essi_dat_cex002_etl'

In [3]:
from decouple import config
from sqlalchemy import create_engine,text
import pandas as pd
from datetime import datetime, timedelta
import time 
import oracledb
import sys
import psycopg2
import numpy as np

In [4]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [6]:
#fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=16", con=connection2)
#fecha= fecha.iloc[0, 0]
#print(fecha)
#
now = datetime.now()
#
#query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=16"
#
#c1= text(query)
#connection2.execute(c1)
fecha="2023-10-24"

#INICIO DEL ESSI

In [10]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
--ATENAMBATENFEC,
to_char(d1.ATENAMBATENFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBATENFEC,
d1.ATENAMBTIPCONCOD,
d1.ATENAMBCSECOD,
d1.CPSCOD,
d1.ATENAMBNUMATECOD,
d1.TIPCONTLEYCOD,
d1.RESATENAMBUCOD,
d1.ORICENASIREFCOD,
d1.CENASIREFCOD,
d1.ATENAMBESTREGCOD,
d1.ATENAMBUSUCRECOD,
--d1.ATENAMBCREFEC,
to_char(d1.ATENAMBCREFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBCREFEC,
d1.ATENAMBUSUMODCOD,
--d1.ATENAMBMODFEC,
to_char(d1.ATENAMBMODFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBMODFEC,
--d1.ATENAMBMODFEC,
to_char(d1.ATENAMBMODFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBMODFEC,
d1.ATENAMBULTREGFEC,
d1.ATENAMBPACSECNUM,

a1.* -- Esto seleccionará todas las columnas de CTDAA10

from {tabla} d1 
left outer join CTDAA10 a1 on a1.ATENAMBORICENASICOD = d1.ATENAMBORICENASICOD
and a1.ATENAMBCENASICOD    = d1.ATENAMBCENASICOD
and a1.ATENAMBNUM    = d1.ATENAMBNUM
where d1.{col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')
"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [ ]:
base1=pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >=TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection3)

In [12]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 895805 entries, 0 to 895804
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   atenambatenfec       895805 non-null  object        
 1   atenambtipconcod     895805 non-null  object        
 2   atenambcsecod        0 non-null       object        
 3   cpscod               895805 non-null  object        
 4   atenambnumatecod     0 non-null       object        
 5   tipcontleycod        895805 non-null  object        
 6   resatenambucod       895805 non-null  object        
 7   oricenasirefcod      0 non-null       object        
 8   cenasirefcod         0 non-null       object        
 9   atenambestregcod     895805 non-null  object        
 10  atenambusucrecod     895805 non-null  object        
 11  atenambcrefec        895805 non-null  object        
 12  atenambusumodcod     483881 non-null  object        
 13  atenambmodfec 

In [13]:
base1.columns

Index(['atenambatenfec', 'atenambtipconcod', 'atenambcsecod', 'cpscod',
       'atenambnumatecod', 'tipcontleycod', 'resatenambucod',
       'oricenasirefcod', 'cenasirefcod', 'atenambestregcod',
       'atenambusucrecod', 'atenambcrefec', 'atenambusumodcod',
       'atenambmodfec', 'atenambmodfec', 'atenambultregfec',
       'atenambpacsecnum', 'atenamboricenasicod', 'atenambcenasicod',
       'atenambnum', 'conddiagcod', 'diagcod', 'atenambdiagord',
       'atenambtipodiagcod', 'atenambcasodiagcod', 'diagatenambaltaflag'],
      dtype='object')

In [14]:
#Eliminamos las columnas que no se usarán aquí: las descripciones y otras 4 más, previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['oricenasirefcod','cenasirefcod']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [15]:
base1 = base1.rename(columns={
    'atenamboricenasicod': 'ori_cas',
    'atenambcenasicod': 'cod_cas',
    'atenambnum': 'ate_num',
    'atenambatenfec': 'ate_fec',
    'atenambtipconcod': 'tip_con',
    'atenambcsecod': 'cod_ser',
    'cpscod': 'cod_cps',
    'atenambnumatecod': 'ate_cod',
    'tipcontleycod': 'ley_cod',
    'resatenambucod': 'res_cod',
    'atenambestregcod': 'est_reg',
    'atenambusucrecod': 'usu_cre',
    'atenambcrefec': 'fec_cre',
    'atenambusumodcod': 'usu_mod',
    'atenambmodfec': 'fec_mod',
    'atenambultregfec': 'fec_ult_reg',
    'atenambpacsecnum': 'pac_sec',
    'conddiagcod': 'cond_diagcod',
    'diagcod': 'diag_cod',
    'atenambdiagord': 'diag_ord',
    'atenambtipodiagcod': 'tip_diag',
    'atenambcasodiagcod': 'caso_diag',
    'diagatenambaltaflag': 'alta_diag'
})

In [16]:
base1.shape

(895805, 24)

In [17]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [19]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 39 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   ori_cas       10 non-null     object             
 1   cod_cas       10 non-null     object             
 2   des_cas       10 non-null     object             
 3   cod_red       10 non-null     object             
 4   des_red       10 non-null     object             
 5   ate_num       10 non-null     float64            
 6   ate_fec       10 non-null     datetime64[ns, UTC]
 7   tip_con       10 non-null     object             
 8   des_tip       10 non-null     object             
 9   cod_ser       0 non-null      object             
 10  des_ser       0 non-null      object             
 11  cod_cps       10 non-null     object             
 12  des_cps       10 non-null     object             
 13  ate_cod       0 non-null      object             
 14  des_ate      

#TRAEMOS TODOS LOS MAESTROS

In [20]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
#id_red,cod_cas,des_cas,cod_red,des_red

tipcon=pd.read_sql_query(f"SELECT cod_tipcon,des_tipcon FROM dim_tipcon", con=connection2)
tipcon['cod_tipcon']=tipcon['cod_tipcon'].str.strip()

servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()

cps = pd.read_sql_query(f"SELECT cod_cps,des_cps FROM dim_cps", con=connection2)
cps['cod_cps']=cps['cod_cps'].str.strip()

numaten = pd.read_sql_query(f"SELECT cod_numaten,des_numaten FROM dim_numaten", con=connection2)
numaten['cod_numaten']=numaten['cod_numaten'].str.strip()

tipleycont = pd.read_sql_query(f"SELECT cod_tipleycont,des_tipleycont FROM dim_tipleycont", con=connection2)
tipleycont['cod_tipleycont']=tipleycont['cod_tipleycont'].str.strip()

resaten = pd.read_sql_query(f"SELECT cod_resaten,des_resaten FROM dim_resaten", con=connection2)
resaten['cod_resaten']=resaten['cod_resaten'].str.strip()

estreg = pd.read_sql_query(f"SELECT cod_est,des_est FROM dim_estreg", con=connection2)
estreg['cod_est']=estreg['cod_est'].str.strip()

#pacsec = pd.read_sql_query(f"SELECT per_sec,id_tipdoc,num_doc FROM dim_pacsec", con=connection2)

condiag = pd.read_sql_query(f"SELECT cod_con,des_con FROM dim_condiag", con=connection2)
condiag['cod_con']=condiag['cod_con'].str.strip()

cie10 = pd.read_sql_query(f"SELECT cod_cie,des_cie FROM dim_cie10", con=connection2)
cie10['cod_cie']=cie10['cod_cie'].str.strip()

tipdx = pd.read_sql_query(f"SELECT cod_tdx,des_tdx FROM dim_tipdx", con=connection2)
tipdx['cod_tdx']=tipdx['cod_tdx'].str.strip()

casdiag = pd.read_sql_query(f"SELECT cod_casdiag, des_casdiag FROM dim_casdiag", con=connection2)
casdiag['cod_casdiag']=casdiag['cod_casdiag'].str.strip()


In [91]:
pacsec.info()
#19957234

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23792262 entries, 0 to 23792261
Data columns (total 3 columns):
 #   Column     Dtype  
---  ------     -----  
 0   per_sec    float64
 1   id_tipdoc  float64
 2   num_doc    object 
dtypes: float64(2), object(1)
memory usage: 544.6+ MB


In [92]:
a=base1.copy()

In [17]:
#base1=a

In [21]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 895805 entries, 0 to 895804
Data columns (total 24 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   ate_fec       895805 non-null  object        
 1   tip_con       895805 non-null  object        
 2   cod_ser       0 non-null       object        
 3   cod_cps       895805 non-null  object        
 4   ate_cod       0 non-null       object        
 5   ley_cod       895805 non-null  object        
 6   res_cod       895805 non-null  object        
 7   est_reg       895805 non-null  object        
 8   usu_cre       895805 non-null  object        
 9   fec_cre       895805 non-null  object        
 10  usu_mod       483881 non-null  object        
 11  fec_mod       895805 non-null  object        
 12  fec_mod       895805 non-null  object        
 13  fec_ult_reg   12318 non-null   datetime64[ns]
 14  pac_sec       895805 non-null  int64         
 15  ori_cas       895

In [22]:
#des_cas, cod_red, des_red
base1=pd.merge(base1,cas_red,how='left',on='cod_cas')
base1=base1.drop("id_red", axis=1)
base1.shape

(895805, 27)

In [23]:
#des_tip
base1['tip_con']=base1['tip_con'].str.strip()
base1=pd.merge(base1,tipcon,how='left',left_on='tip_con',right_on='cod_tipcon')
base1=base1.rename(columns={"des_tipcon":"des_tip"})
base1 = base1.drop("cod_tipcon", axis=1)
base1.shape

(895805, 28)

In [24]:
#des_ser
base1['cod_ser']=base1['cod_ser'].str.strip()
base1=pd.merge(base1,servicios,how='left',on='cod_ser')
base1.shape

(895805, 29)

In [25]:
#des_cps
base1['cod_cps']=base1['cod_cps'].str.strip()
base1=pd.merge(base1,cps,how='left',on='cod_cps')
base1.shape

(895805, 30)

In [26]:
#des_ate
base1['ate_cod']=base1['ate_cod'].str.strip()
base1=pd.merge(base1,numaten,how='left',left_on='ate_cod',right_on='cod_numaten')
base1=base1.rename(columns={"des_numaten":"des_ate"})
base1 = base1.drop("cod_numaten", axis=1)
base1.shape

(895805, 31)

In [27]:
#des_ley
base1['ley_cod']=base1['ley_cod'].str.strip()
base1=pd.merge(base1,tipleycont,how='left',left_on='ley_cod',right_on='cod_tipleycont')
base1=base1.rename(columns={"des_tipleycont":"des_ley"})
base1 = base1.drop("cod_tipleycont", axis=1)
base1.shape

(895805, 32)

In [28]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 895805 entries, 0 to 895804
Data columns (total 32 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   ate_fec       895805 non-null  object        
 1   tip_con       895805 non-null  object        
 2   cod_ser       0 non-null       object        
 3   cod_cps       895805 non-null  object        
 4   ate_cod       0 non-null       object        
 5   ley_cod       895805 non-null  object        
 6   res_cod       895805 non-null  object        
 7   est_reg       895805 non-null  object        
 8   usu_cre       895805 non-null  object        
 9   fec_cre       895805 non-null  object        
 10  usu_mod       483881 non-null  object        
 11  fec_mod       895805 non-null  object        
 12  fec_mod       895805 non-null  object        
 13  fec_ult_reg   12318 non-null   datetime64[ns]
 14  pac_sec       895805 non-null  int64         
 15  ori_cas       895

In [29]:
#des_res
base1['res_cod']=base1['res_cod'].str.strip()
base1=pd.merge(base1,resaten,how='left',left_on='res_cod',right_on='cod_resaten')
base1=base1.rename(columns={"des_resaten":"des_res"})
base1 = base1.drop("cod_resaten", axis=1)
base1.shape

(895805, 33)

In [30]:
#des_est
base1['est_reg']=base1['est_reg'].str.strip()
base1=pd.merge(base1,estreg,how='left',left_on='est_reg',right_on='cod_est')
base1 = base1.drop("cod_est", axis=1)
base1.shape

(895805, 34)

In [31]:
b=base1.copy()

In [28]:
#base1=b

In [103]:
#cod_tdi, num_doc
base1=pd.merge(base1,pacsec,how="left",left_on='pac_sec',right_on='per_sec')
base1=base1.rename(columns={"id_tipdoc":"cod_tdi"})
base1["num_doc"]=base1['num_doc'].str.strip()
base1=base1.drop("per_sec", axis=1)
base1.shape

(236971, 35)

In [104]:
c=base1.copy()

In [31]:
base1=c

In [120]:
base1=c

In [32]:
# columnas_de_interes = ['cod_tdi', 'num_doc','pac_sec']
# filas_con_cod_tdi_nulo = base1[base1['cod_tdi'].isnull()][columnas_de_interes]
# filas_con_cod_tdi_nulo

In [32]:
#des_condiag
base1['cond_diagcod']=base1['cond_diagcod'].str.strip()
base1=pd.merge(base1,condiag,how='left',left_on='cond_diagcod',right_on='cod_con')
base1=base1.rename(columns={"des_con":"des_condiag"})
base1 = base1.drop("cod_con", axis=1)
base1.shape

(895805, 35)

In [33]:
#des_diag
base1['diag_cod']=base1['diag_cod'].str.strip()
base1=pd.merge(base1,cie10,how='left',left_on='diag_cod',right_on='cod_cie')
base1=base1.rename(columns={"des_cie":"des_diag"})
base1 = base1.drop("cod_cie", axis=1)
base1.shape

(895811, 36)

In [36]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 895811 entries, 0 to 895810
Data columns (total 37 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   ate_fec       895811 non-null  object        
 1   tip_con       895811 non-null  object        
 2   cod_ser       0 non-null       object        
 3   cod_cps       895811 non-null  object        
 4   ate_cod       0 non-null       object        
 5   ley_cod       895811 non-null  object        
 6   res_cod       895811 non-null  object        
 7   est_reg       895811 non-null  object        
 8   usu_cre       895811 non-null  object        
 9   fec_cre       895811 non-null  object        
 10  usu_mod       483883 non-null  object        
 11  fec_mod       895811 non-null  object        
 12  fec_mod       895811 non-null  object        
 13  fec_ult_reg   12318 non-null   datetime64[ns]
 14  pac_sec       895811 non-null  int64         
 15  ori_cas       895

In [35]:
#des_tipdiag
#base1['tip_diag'] = base1['tip_diag'].astype('Int64')
#base1['tip_diag'] = base1['tip_diag'].astype(str)
base1['tip_diag']=base1['tip_diag'].str.strip()
base1=pd.merge(base1,tipdx,how='left',left_on='tip_diag',right_on='cod_tdx')
base1=base1.rename(columns={"des_tdx":"des_tipdiag"})
base1 = base1.drop("cod_tdx", axis=1)
base1.shape

(895811, 37)

In [37]:
#caso_diagdes
# base1['caso_diag'] = base1['caso_diag'].astype('Int64')
# base1['caso_diag'] = base1['caso_diag'].astype(str)
base1['caso_diag']=base1['caso_diag'].str.strip()
base1=pd.merge(base1,casdiag,how='left',left_on='caso_diag',right_on='cod_casdiag')
base1=base1.rename(columns={"des_casdiag":"caso_diagdes"})
base1 = base1.drop("cod_casdiag", axis=1)
base1.shape

(895811, 38)

In [127]:
# base1['diag_ord'] = base1['diag_ord'].astype('Int64')
# base1['ate_num'] = base1['ate_num'].astype('Int64')
# base1['pac_sec'] = base1['pac_sec'].astype('Int64')
# base1['alta_diag'] = base1['alta_diag'].astype('Int64')
# base1['cod_tdi'] = base1['cod_tdi'].astype('Int64')


In [128]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 236973 entries, 0 to 236972
Data columns (total 39 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   ori_cas       236954 non-null  object
 1   cod_cas       236954 non-null  object
 2   ate_num       236954 non-null  Int64 
 3   ate_fec       236973 non-null  object
 4   tip_con       236973 non-null  object
 5   cod_ser       0 non-null       object
 6   cod_cps       236973 non-null  object
 7   ate_cod       0 non-null       object
 8   ley_cod       236973 non-null  object
 9   res_cod       236973 non-null  object
 10  est_reg       236973 non-null  object
 11  usu_cre       236973 non-null  object
 12  fec_cre       236973 non-null  object
 13  usu_mod       127119 non-null  object
 14  fec_mod       236973 non-null  object
 15  fec_ult_reg   3571 non-null    object
 16  pac_sec       236973 non-null  Int64 
 17  cond_diagcod  236954 non-null  object
 18  diag_cod      236954 non

In [164]:
base2.columns

Index(['ori_cas', 'cod_cas', 'des_cas', 'cod_red', 'des_red', 'ate_num',
       'ate_fec', 'tip_con', 'des_tip', 'cod_ser', 'des_ser', 'cod_cps',
       'des_cps', 'ate_cod', 'des_ate', 'ley_cod', 'des_ley', 'res_cod',
       'des_res', 'est_reg', 'des_est', 'usu_cre', 'fec_cre', 'usu_mod',
       'fec_mod', 'fec_ult_reg', 'cod_tdi', 'num_doc', 'pac_sec',
       'cond_diagcod', 'des_condiag', 'diag_cod', 'des_diag', 'diag_ord',
       'tip_diag', 'des_tipdiag', 'caso_diag', 'caso_diagdes', 'alta_diag'],
      dtype='object')

In [165]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [166]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [167]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [168]:
base.head()

,des_res,cond_diagcod,des_condiag,est_reg,pac_sec,res_cod,ate_cod,cod_cas,des_red,cod_ser,...,cod_tdi,ley_cod,diag_ord,des_ley,usu_cre,alta_diag,fec_ult_reg,fec_mod,diag_cod,ate_num
0,PROCEDIMIENTOS,2,SALIDA,1,7058437,11,None,001,RED PRESTACIONAL REBAGLIATI ...,None,...,1,00,1,NO CORRESPONDE,09563306,0,None,2023-10-25,M48.0,14389497
1,PROCEDIMIENTOS,2,SALIDA,1,7058437,11,None,001,RED PRESTACIONAL REBAGLIATI ...,None,...,1,00,2,NO CORRESPONDE,09563306,0,None,2023-10-25,M89.9,14389497
2,PROCEDIMIENTOS,2,SALIDA,1,7058437,11,None,001,RED PRESTACIONAL REBAGLIATI ...,None,...,1,00,3,NO CORRESPONDE,09563306,0,None,2023-10-25,H54.4,14389497
3,PROCEDIMIENTOS,2,SALIDA,1,7058437,11,None,001,RED PRESTACIONAL REBAGLIATI ...,None,...,1,00,4,NO CORRESPONDE,09563306,0,None,2023-10-25,F41.9,14389497
4,ALTA,2,SALIDA,1,4389509,01,None,001,RED PRESTACIONAL REBAGLIATI ...,None,...,1,00,1,NO CORRESPONDE,73011999,0,None,2023-10-25,L98.4,14461040


In [135]:
base.columns

Index(['des_res', 'cond_diagcod', 'des_condiag', 'est_reg', 'pac_sec',
       'res_cod', 'ate_cod', 'cod_cas', 'des_red', 'cod_ser', 'des_est',
       'usu_mod', 'cod_red', 'ate_fec', 'des_cps', 'tip_diag', 'num_doc',
       'caso_diagdes', 'des_ser', 'des_ate', 'fec_cre', 'des_cas', 'des_tip',
       'des_tipdiag', 'caso_diag', 'ori_cas', 'des_diag', 'tip_con', 'cod_cps',
       'cod_tdi', 'ley_cod', 'diag_ord', 'des_ley', 'usu_cre', 'alta_diag',
       'fec_ult_reg', 'fec_mod', 'diag_cod', 'ate_num'],
      dtype='object')

In [136]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [ ]:
base1['diag_ord'] = base1['diag_ord'].astype('Int64')
base1['ate_num'] = base1['ate_num'].astype('Int64')
base1['pac_sec'] = base1['pac_sec'].astype('Int64')
base1['alta_diag'] = base1['alta_diag'].astype('Int64')
base1['cod_tdi'] = base1['cod_tdi'].astype('Int64')

In [156]:
# #Query para obtener el número máximo de caracteres de una columna
# base["alta_diag"]=base["alta_diag"].astype(str)
# base["alta_diag"].str.len().max()

4

In [162]:
base1=base1.replace('<NA>', None)

In [169]:
base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=10000)

23973

In [53]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3)
print("Proceso 1.2 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.2 completado satisfactoriamente en 867.212 segundos


In [54]:
connection1.close()
connection2.close()
connection3.close()

In [55]:
engine1.dispose()
engine2.dispose()
engine3.dispose()